# Scrapping the Indian Econometric Society
**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append






### Mounting Google Drive as a working director

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installing and loading all dependencies and libraries needed

In [2]:
#Install necessary dependencies and libraries
!pip install waybackpy
!pip install googlesearch-python
!pip install google-api-python-client
!pip install beautifulsoup4
!pip install fake_useragent #necessary for google search rapid automation
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!apt-get install -y wget
!pip install requests-html

import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging


from waybackpy import WaybackMachineSaveAPI
from googlesearch import search
from googleapiclient.discovery import build
import time
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import logging

#from requests_html import HTMLSession
#from requests_html import AsyncHTMLSession
import pandas as pd
import logging
import asyncio
import nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.3/486.3 kB 24.3 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [61.7 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,199 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubunt

# Scrapping ISI: 3rd attempt: PDF MINER.SIX

## Extracting text

In [3]:
!pip install pdfminer.six
from pdfminer.high_level import extract_text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 33.8 MB/s eta 0:00:00


In [7]:
# Specify the path to your PDF in Google Drive
pdf_path = '/content/drive/My Drive/ISI/2021_removed.pdf'

In [8]:
text = extract_text(pdf_path)
print(text)


16th Annual Conference on Economic Growth and 
Development 

20-22 December, 2021 
Indian Statistical Institute, Delhi 

Conference Programme 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Day 1: 20 December, Monday 

8.00 - 8.15 (IST) (21.30 - 21.45 previous night EST):  

Welcome and Opening Remarks: Debasis Mishra (Indian Statistical Institute, Delhi) 

8.15 - 9.30 (IST) (21.45 - 23.00 previous night EST): Plenary Session 1 

Raymond Fisman (Boston University): “The Costs and Benefits of Social Ties” 
Chair: Abhiroop Mukhopadhyay (Indian Statistical Institute, Delhi) 

9.30 - 10.00 (IST) (23.00 - 23.30 previous night EST): Break 

10.00 - 11.45 (IST) (23.30 previous night - 1.15 EST): Parallel Sessions 1 

1(a) Education I 
1(b) Finance I: Housing and Corporate Finance  
1(c) Health I: COVID 19 
1(d) Labour I 
1(e) Public Economics I 
1(f) Political Economy I 
1(g) Environment I 
1(h) Communication and Information 
1(i) IO Theory I 
1(j) Monetary Policy 

18.00 - 19.45 (IST) (7.30 - 9.15 EST): 

In [ ]:
import re
import pandas as pd
from pdfminer.high_level import extract_text
import spacy

# Load spaCy model with entity recognition
nlp = spacy.load("en_core_web_sm")

def extract_pdf_text(pdf_path):
    """Extract text from a PDF file."""
    return extract_text(pdf_path)

def parse_member_entry(entry):
    """
    Parse individual entries to extract Name, Affiliation, Address, and Email.
    """
    # Initialize fields
    name = None
    affiliation = None
    address = None
    email = None

    # NLP processing
    doc = nlp(entry)

    # Extract email
    email_match = re.search(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", entry)
    email = email_match.group(0) if email_match else None

    # Extract name using PERSON entity and sentence structure
    for ent in doc.ents:
        if ent.label_ == "PERSON" and len(ent.text.split()) > 1:
            name = ent.text
            break

    # Extract affiliation heuristically (e.g., organization names or professional titles)
    for ent in doc.ents:
        if ent.label_ in ["ORG", "WORK_OF_ART"] and not affiliation:
            affiliation = ent.text

    # Extract address as remaining text after email and name
    remaining_text = entry
    if email:
        remaining_text = remaining_text.replace(email, "").strip()
    if name:
        remaining_text = remaining_text.replace(name, "").strip()

    # Use heuristics for address extraction
    address_candidates = [
        line.strip()
        for line in remaining_text.split("\n")
        if len(line.strip()) > 5 and not re.search(r"[0-9]+@", line)
    ]
    address = " ".join(address_candidates) if address_candidates else None

    return {"Name": name, "Affiliation": affiliation, "Address": address, "Email": email}

def parse_members_list(text):
    """
    Split text into individual member entries and extract structured information.
    """
    data = []
    entries = text.split("\n\n")  # Split by double newline for potential separation

    for entry in entries:
        if len(entry.strip()) < 20 or "Page" in entry:
            continue  # Skip irrelevant or very short entries
        member_info = parse_member_entry(entry)
        if member_info["Name"]:  # Only include if a valid name is found
            data.append(member_info)

    return data

def process_pdf_to_df(pdf_path):
    """
    Extract text from a PDF and process it into a structured DataFrame.
    """
    text = extract_pdf_text(pdf_path)
    members_data = parse_members_list(text)

    # Create DataFrame
    df = pd.DataFrame(members_data)

    # Remove rows with all None values and ensure meaningful data
    df.dropna(how="all", inplace=True)

    # Reorder and clean columns
    df = df[["Name", "Affiliation", "Address", "Email"]]
    return df

# Load the PDF file
pdf_path = "/content/drive/My Drive/IES.pdf"  # Replace with your file path
df = process_pdf_to_df(pdf_path)

# Save or display the DataFrame
df.to_csv("members_list_clean.csv", index=False)
print(df.head())


                 Name         Affiliation  \
0         S. N.  Name                None   
1         S. N.  Name                None   
2           Badr Abdi              L.2169   
3          S. Acharya  Rajasthan \nL.2080   
4  Rajesh Kumar Verma                None   

                                             Address Email  
0                                of the Life Members  None  
1                                of the Life Members  None  
2  L.2169 (96-97) 83/III Jai Jawan Colony JLN Mar...  None  
3  L.1120 (89-90) Shabd. Honorary Professor 33, S...  None  
4  L.2170 (91-92) 93-Nalapar, Purbia Tola Etawah, UP  None  


In [ ]:
print(df)

                     Name              Affiliation  \
0             S. N.  Name                     None   
1             S. N.  Name                     None   
2               Badr Abdi                   L.2169   
3              S. Acharya       Rajasthan \nL.2080   
4      Rajesh Kumar Verma                     None   
..                    ...                      ...   
957  Yellukunte Extension        Bommanahalli Post   
958     Harikrishnan K.S.            IIT Kharagpur   
959         Apoorva Verma                     None   
960   Bhanu Pratap Pandey                     None   
961           Manu Sharma  Department of Economics   

                                               Address  \
0                                  of the Life Members   
1                                  of the Life Members   
2    L.2169 (96-97) 83/III Jai Jawan Colony JLN Mar...   
3    L.1120 (89-90) Shabd. Honorary Professor 33, S...   
4    L.2170 (91-92) 93-Nalapar, Purbia Tola Etawah, UP   
.. 

Clean the df

In [ ]:
def clean_and_update_affiliations(df):
    """
    Clean and update the DataFrame:
    1. Drop rows where 'Name' is 'S. N. Name'.
    2. Append university names from 'Address' to 'Affiliation' where applicable.
    """
    # Drop rows where 'Name' is 'S. N. Name'
    df = df[df['Name'] != 'S. N. Name']

    # NLP model for processing Address
    nlp = spacy.load("en_core_web_sm")

    # Function to find and append university names
    def append_university(row):
        address = row['Address']
        affiliation = row['Affiliation'] or ""

        if pd.isna(address):
            return affiliation  # If no address, return existing affiliation

        doc = nlp(address)
        universities = [ent.text for ent in doc.ents if "university" in ent.text.lower()]

        if universities:
            # Append found universities to the affiliation
            affiliation = f"{affiliation}, {', '.join(universities)}" if affiliation else ", ".join(universities)

        return affiliation

    # Update the 'Affiliation' column
    df['Affiliation'] = df.apply(append_university, axis=1)

    return df

# Clean and update the DataFrame
df_cleaned = clean_and_update_affiliations(df)

# Save or display the updated DataFrame
df_cleaned.to_csv("updated_members_list.csv", index=False)
print(df_cleaned.head())


                 Name         Affiliation  \
0         S. N.  Name                       
1         S. N.  Name                       
2           Badr Abdi              L.2169   
3          S. Acharya  Rajasthan \nL.2080   
4  Rajesh Kumar Verma                       

                                             Address Email  
0                                of the Life Members  None  
1                                of the Life Members  None  
2  L.2169 (96-97) 83/III Jai Jawan Colony JLN Mar...  None  
3  L.1120 (89-90) Shabd. Honorary Professor 33, S...  None  
4  L.2170 (91-92) 93-Nalapar, Purbia Tola Etawah, UP  None  


In [ ]:
df_cleaned = df

In [ ]:
# Drop rows where 'Name' is 'S. N. Name'
df = df[df['Name'] != 'S. N.  Name']

In [ ]:
print(df)

                          Name              Affiliation  \
2                    Badr Abdi                   L.2169   
3                   S. Acharya       Rajasthan \nL.2080   
4           Rajesh Kumar Verma                     None   
5    F.A. Saiyed Bagh E Ahmedi                     None   
6                 Ashok Parikh                    L.829   
..                         ...                      ...   
957       Yellukunte Extension        Bommanahalli Post   
958          Harikrishnan K.S.            IIT Kharagpur   
959              Apoorva Verma                     None   
960        Bhanu Pratap Pandey                     None   
961                Manu Sharma  Department of Economics   

                                               Address  \
2    L.2169 (96-97) 83/III Jai Jawan Colony JLN Mar...   
3    L.1120 (89-90) Shabd. Honorary Professor 33, S...   
4    L.2170 (91-92) 93-Nalapar, Purbia Tola Etawah, UP   
5    L.313 (74-75) 9, Roshan Park Society Jivaraj P...   
6

Save dataset

In [ ]:
# Save or display the updated DataFrame
df.to_csv("IES.csv", index=False)
print(df.head())


                        Name         Affiliation  \
2                  Badr Abdi              L.2169   
3                 S. Acharya  Rajasthan \nL.2080   
4         Rajesh Kumar Verma                None   
5  F.A. Saiyed Bagh E Ahmedi                None   
6               Ashok Parikh               L.829   

                                             Address Email  
2  L.2169 (96-97) 83/III Jai Jawan Colony JLN Mar...  None  
3  L.1120 (89-90) Shabd. Honorary Professor 33, S...  None  
4  L.2170 (91-92) 93-Nalapar, Purbia Tola Etawah, UP  None  
5  L.313 (74-75) 9, Roshan Park Society Jivaraj P...  None  
6  L.829 (85-86) Department of Economics Universi...  None  


# Merging

In [ ]:
import pandas as pd

# List of CSV files to load from the Google Drive folder "IES"
csv_files = [
    '/content/drive/My Drive/IES/IES1-294.csv',
    '/content/drive/My Drive/IES/IES311-594.csv',
    '/content/drive/My Drive/IES/IES595-852.csv',
    '/content/drive/My Drive/IES/IES854-980.csv',
    '/content/drive/My Drive/IES/IES981-1096.csv',
    '/content/drive/My Drive/IES/IES1097-1288.csv'
]

# Load each CSV file and check columns
dataframes = []
for file in csv_files:
    df = pd.read_csv(file)
    print(f"Columns in {file}: {list(df.columns)}")  # Check columns in each file
    dataframes.append(df)

Columns in /content/drive/My Drive/IES/IES1-294.csv: ['Number of member', 'ID', 'Name', 'Affiliation', 'Address', 'Email', 'misc1', 'misc2', 'misc3', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']
Columns in /content/drive/My Drive/IES/IES311-594.csv: ['Number of member', 'ID', 'Name', 'Affiliation', 'Address', 'Email', 'misc1', 'misc2', 'misc3', 'Unnamed: 9']
Columns in /content/drive/My Drive/IES/IES595-852.csv: ['Number of member', 'ID', 'Name', 'Affiliation', 'Address', 'Email', 'misc1', 'misc2', 'misc3', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12']
Columns in /content/drive/My Drive/IES/IES854-980.csv: ['Number of member', 'ID', 'Name', 'Affiliation', 'Address', 'Email', 'misc1', 'misc2', 'misc3', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']
Columns in /content/drive/My Drive/IES/IES981-1096.csv: ['Number of member', 'ID', 'Name', 'Affiliation', 'Address', 'Email', 'misc1', 'misc2', 'misc3', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnam

In [ ]:
# Concatenate all DataFrames
combined_df = pd.concat(dataframes, ignore_index=True)

# Save the combined DataFrame as a CSV file to the "IES" folder on Google Drive
output_path = '/content/drive/My Drive/IES/combined_IES.csv'
combined_df.to_csv(output_path, index=False)

print(f"Combined CSV saved to {output_path}")

Combined CSV saved to /content/drive/My Drive/IES/combined_IES.csv
